## Quality control — “path overlay visualization”

haven’t yet verified whether smoothed paths actually follow the video.

Implement a simple script to overlay hand trajectories over the video:
- Last x frames of path (tail trace)
- Optionally color-code velocity (blue=slow, red=fast)
- Save as annotated video for visual inspection
- Check whether smoothed paths lag too much or deviate from actual motion.
- Priority: High — validating your core metric computation is critical before doing more complex modeling.

In [37]:
# imports
import numpy as np
import pandas as pd
import os
import cv2
from tqdm import tqdm

In [31]:
video_paths = os.listdir('Study1_Videos')
processed_df_paths = os.listdir('processed_dataframes')

processed_df_names_10fps = [f for f in processed_df_paths if '10fps' in f]
processed_df_names_30fps = [f for f in processed_df_paths if '30fps' in f]


In [14]:
df_name = processed_df_names_10fps[0]
df = pd.read_pickle(os.path.join('processed_dataframes', df_name))
df[df['hand_label'] == 'Right']

,frame,hand_label,hand_score,bbox_center,palm_center,landmarks,frame_diff,segment_id,cx_smooth,cy_smooth,disp,disp_filtered,disp_raw
1,0,Right,0.987475,"(559, 610)","(516, 598)","[{'id': 0, 'coord': (0.2249636948108673, 0.499...",NaN,0,558.200000,610.457143,NaN,NaN,NaN
3,3,Right,0.982245,"(581, 642)","(552, 630)","[{'id': 0, 'coord': (0.2338619977235794, 0.527...",3.0,0,583.400000,641.571429,40.039215,40.039215,38.832976
4,6,Right,0.979613,"(606, 668)","(584, 657)","[{'id': 0, 'coord': (0.2457282394170761, 0.558...",3.0,0,603.600000,666.542857,32.118721,32.118721,36.069378
7,9,Right,0.984403,"(618, 683)","(600, 673)","[{'id': 0, 'coord': (0.25319212675094604, 0.57...",3.0,0,619.457143,684.628571,24.052901,24.052901,19.209373
9,12,Right,0.984056,"(629, 699)","(625, 691)","[{'id': 0, 'coord': (0.2624625563621521, 0.600...",3.0,0,627.885714,698.228571,16.000026,16.000026,19.416488
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9369,18090,Right,0.918975,"(708, 357)","(686, 345)","[{'id': 0, 'coord': (0.36986786127090454, 0.27...",3.0,57,707.400000,356.314286,39.512809,39.512809,40.521599
9370,18093,Right,0.959093,"(716, 385)","(700, 374)","[{'id': 0, 'coord': (0.3774813413619995, 0.311...",3.0,57,716.171429,384.228571,29.259961,29.259961,29.120440
9371,18096,Right,0.987816,"(713, 399)","(703, 400)","[{'id': 0, 'coord': (0.3818809986114502, 0.326...",3.0,57,709.485714,398.657143,15.902278,15.902278,14.317821
9372,18099,Right,0.993462,"(689, 405)","(683, 407)","[{'id': 0, 'coord': (0.36984434723854065, 0.33...",3.0,57,695.542857,408.428571,17.025983,17.025983,24.738634


In [51]:
def draw_hand_trajectories(video_path, fps=30, tail_length=30, velocity_overlay=True):
        # --- 1. Load Dataframe based on FPS ---
    if fps == 30:
        pkl_path = os.path.join('processed_dataframes', 'hand_tracking_' + video_path.split('/')[-1].replace('.mp4', '_30fps_processed.pkl'))
    elif fps == 10:
        pkl_path = os.path.join('processed_dataframes', 'hand_tracking_' + video_path.split('/')[-1].replace('.mp4', '_10fps_processed.pkl'))
    else:
        raise ValueError("fps must be either 10 or 30")

    if not os.path.exists(pkl_path):
        raise FileNotFoundError(f"Processed dataframe not found at: {pkl_path}")
        
    df = pd.read_pickle(pkl_path)
    
    # --- 2. Calculate True Velocity and Handle NaNs (No change) ---
    dt = 1.0 / fps
    df['velocity'] = df['disp_filtered'] / dt
    df['velocity'] = df['velocity'].replace([np.inf, -np.inf], np.nan) 

    # --- 3. Setup Video I/O (No change) ---
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)

    output_video_path = video_path.replace('.mp4', f'_annotated_vel_{fps}fps.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, video_fps, (frame_width, frame_height))

    # --- 4. Velocity and Color Mapping Setup (No change) ---
    default_color = (0, 255, 0)
    # Calculate max_vel and define get_color_from_velocity function (Code omitted for brevity)
    if velocity_overlay:
        valid_velocities = df['velocity'].dropna()
        max_vel = valid_velocities.abs().quantile(0.95) if not valid_velocities.empty and valid_velocities.abs().quantile(0.95) != 0 else 1.0
        
        def get_color_from_velocity(velocity):
            speed = abs(velocity)
            norm_speed = np.clip(speed / max_vel, 0, 1)
            B = int(255 * (1 - norm_speed))
            R = int(255 * norm_speed)
            G = 0
            return (B, G, R)

    # --- 5. Process Video Frame by Frame (Core Logic Change) ---
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Annotating video: {video_path}")
    
    with tqdm(total=total_frames, unit="frames", desc="Processing Frames") as pbar: 
        current_video_frame = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break # End of video
            
            # This frame is ALWAYS processed (read and eventually written)
            
            for hand_label in ['Right', 'Left']:
                # The path data should now include all relevant frames from the dataframe, 
                # even if the *current* frame is missing from df.
                tail_start_frame = max(0, current_video_frame - tail_length)
                
                # Get the history path data for the current hand: 
                # All points (including the current frame's point, if available) in the tail range
                path_data = df[
                    (df['hand_label'] == hand_label) & 
                    (df['frame'] >= tail_start_frame) & 
                    (df['frame'] <= current_video_frame)
                ].sort_values(by='frame')

                # Ensure we have at least 2 points to draw a line
                if len(path_data) > 1:
                    # --- A. Draw the Tail Trace ---
                    for i in range(1, len(path_data)):
                        segment_disp = path_data['disp_filtered'].iloc[i]
                        segment_vel = path_data['velocity'].iloc[i]

                        # **CRITICAL FIX**: Only draw lines when displacement is > 0 and not NaN
                        if pd.isna(segment_disp) or segment_disp <= 0:
                            continue 
                            
                        # Coordinates must exist because the row exists in path_data
                        p1 = (int(path_data['cx_smooth'].iloc[i-1]), int(path_data['cy_smooth'].iloc[i-1]))
                        p2 = (int(path_data['cx_smooth'].iloc[i]), int(path_data['cy_smooth'].iloc[i]))
                        
                        # Determine color for the line segment
                        line_color = get_color_from_velocity(segment_vel) if velocity_overlay else default_color
                        
                        cv2.line(frame, p1, p2, line_color, 2)
                        cv2.circle(frame, p2, 3, line_color, -1)


            # Write the annotated frame (even if no hands were drawn on it)
            # ---- Frame Display ----
            cv2.putText(frame, f'Frame: {current_video_frame}', (10, 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
            
            out.write(frame)
            current_video_frame += 1
            pbar.update(1)


    # --- 6. Cleanup and Return (No change) ---
    cap.release()
    out.release()
    
    return output_video_path

In [52]:
annotated_path = draw_hand_trajectories('Study1_Videos/2024-01-15_16-17-19.mp4', fps=30)

Annotating video: Study1_Videos/2024-01-15_16-17-19.mp4


Processing Frames: 100%|██████████| 9535/9535 [03:19<00:00, 47.77frames/s]


In [56]:
def compare_trajectories(video_path, tail_length=30):
    """
    Overlays and compares hand trajectories from both 10 fps and 30 fps processed dataframes 
    on the original video.

    Args:
        video_path (str): Path to the input video file (e.g., 'example.mp4').
        tail_length (int): Number of previous frames to trace for the hand's path (tail).

    Returns:
        str: Path to the saved comparison video file.
    """
    
    # --- 1. Load and Prepare Dataframes ---
    
    data_sources = {}
    
    # 10 fps Data
    pkl_path_10 = os.path.join('processed_dataframes', 'hand_tracking_' + video_path.split('/')[-1].replace('.mp4', '_10fps_processed.pkl'))
    if os.path.exists(pkl_path_10):
        df_10 = pd.read_pickle(pkl_path_10)
        df_10['velocity'] = df_10['disp_filtered'] / (1.0 / 10)
        data_sources['10fps'] = {'df': df_10, 'color': (255, 0, 0)} # Blue
    else:
        print(f"Warning: 10 fps dataframe not found at: {pkl_path_10}")

    # 30 fps Data
    pkl_path_30 = os.path.join('processed_dataframes', 'hand_tracking_' + video_path.split('/')[-1].replace('.mp4', '_30fps_processed.pkl'))
    if os.path.exists(pkl_path_30):
        df_30 = pd.read_pickle(pkl_path_30)
        df_30['velocity'] = df_30['disp_filtered'] / (1.0 / 30)
        data_sources['30fps'] = {'df': df_30, 'color': (0, 0, 255)} # Red
    else:
        print(f"Warning: 30 fps dataframe not found at: {pkl_path_30}")

    if not data_sources:
        raise FileNotFoundError("Neither 10 fps nor 30 fps dataframes were found. Cannot compare.")

    # --- 2. Setup Video I/O ---
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)

    output_video_path = video_path.replace('.mp4', '_comparison.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, video_fps, (frame_width, frame_height))

    # --- 3. Process Video Frame by Frame (with TQDM) ---
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Starting comparison annotation for video: {video_path}")
    
    # Define a single color function for the velocity-less comparison
    # We use fixed colors (Blue for 10fps, Red for 30fps) instead of velocity coding
    # to clearly distinguish the two tracking rates.
    
    with tqdm(total=total_frames, unit="frames", desc="Processing Frames") as pbar: 
        current_video_frame = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Draw trajectories for EACH processed rate (10fps and 30fps)
            for label, data in data_sources.items():
                df_rate = data['df']
                line_color = data['color'] # Blue or Red
                
                for hand_label in ['Right', 'Left']:
                    tail_start_frame = max(0, current_video_frame - tail_length)
                    
                    # Query data for the current rate and hand within the tail range
                    path_data = df_rate[
                        (df_rate['hand_label'] == hand_label) & 
                        (df_rate['frame'] >= tail_start_frame) & 
                        (df_rate['frame'] <= current_video_frame)
                    ].sort_values(by='frame')

                    if len(path_data) > 1:
                        # Draw the Tail Trace
                        for i in range(1, len(path_data)):
                            segment_disp = path_data['disp_filtered'].iloc[i]

                            # Only draw lines when displacement is > 0 and not NaN
                            if pd.isna(segment_disp) or segment_disp <= 0:
                                continue 
                                
                            p1 = (int(path_data['cx_smooth'].iloc[i-1]), int(path_data['cy_smooth'].iloc[i-1]))
                            p2 = (int(path_data['cx_smooth'].iloc[i]), int(path_data['cy_smooth'].iloc[i]))
                            
                            # Draw the line using the fixed color for the rate
                            cv2.line(frame, p1, p2, line_color, 2)
                            cv2.circle(frame, p2, 3, line_color, -1)
                            
                            # Add a label near the point (optional, but helpful for debugging)
                            # if i == len(path_data) - 1:
                            #     cv2.putText(frame, f'{label[0]}', p2, cv2.FONT_HERSHEY_SIMPLEX, 0.5, line_color, 1, cv2.LINE_AA)

            # --- 4. Add Legend (for clarity) ---
            # Define legend positions
            cv2.line(frame, (20, 30), (50, 30), data_sources.get('10fps', {}).get('color', (100, 100, 100)), 3)
            cv2.putText(frame, '10 fps', (60, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.line(frame, (20, 60), (50, 60), data_sources.get('30fps', {}).get('color', (100, 100, 100)), 3)
            cv2.putText(frame, '30 fps', (60, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

            # ---- Frame Display ----
            cv2.putText(frame, f'Frame: {current_video_frame}', (10, 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
            # Write the annotated frame
            out.write(frame)
            current_video_frame += 1
            pbar.update(1)

    # --- 5. Cleanup and Return ---
    cap.release()
    out.release()
    
    return output_video_path

In [64]:
annotated_path = compare_trajectories('Study1_Videos/2024-01-18_13-11-31.mp4')

Starting comparison annotation for video: Study1_Videos/2024-01-18_13-11-31.mp4


Processing Frames: 100%|█████████▉| 11761/11773 [04:38<00:00, 42.30frames/s]


In [ ]:
df = pd.read_pickle(os.path.join('output_dataframes', 'hand_tracking_2024-01-23_08-23-33_10fps.pkl'))

In [63]:
df['landmarks'].iloc[0]

[{'id': 0,
  'coord': (0.5916430950164795, 0.5127965807914734, -1.372092270912617e-07)},
 {'id': 1,
  'coord': (0.5629996061325073, 0.5119854807853699, -0.009370381943881512)},
 {'id': 2,
  'coord': (0.5416164398193359, 0.5162436962127686, -0.021821018308401108)},
 {'id': 3,
  'coord': (0.5279139876365662, 0.5342140197753906, -0.03266414627432823)},
 {'id': 4,
  'coord': (0.5194789171218872, 0.5524727702140808, -0.04464377835392952)},
 {'id': 5,
  'coord': (0.5488148331642151, 0.4740365147590637, -0.03888876363635063)},
 {'id': 6,
  'coord': (0.5255122780799866, 0.5032265782356262, -0.057038817554712296)},
 {'id': 7,
  'coord': (0.5166718363761902, 0.5381742715835571, -0.06662441045045853)},
 {'id': 8,
  'coord': (0.5131765007972717, 0.5658705830574036, -0.07201630622148514)},
 {'id': 9,
  'coord': (0.5658946633338928, 0.485118567943573, -0.04251503571867943)},
 {'id': 10,
  'coord': (0.543163001537323, 0.5117312073707581, -0.05992449074983597)},
 {'id': 11,
  'coord': (0.5329461693763